In [1]:
import boto3

s3 = boto3.client("s3")
print("caller identity ok")  # just to show code runs

# list buckets (if this errors, your role is super restricted)
resp = s3.list_buckets()
print([b["Name"] for b in resp.get("Buckets", [])])

caller identity ok
['amazon-sagemaker-313730224547-us-east-1-5epobvmonaep0n']


In [2]:
import sys
sys.path.append("./")

In [3]:
from pathlib import Path
from src.s3_fetch import download_applicant_data

download_applicant_data(
    bucket_name="amazon-sagemaker-313730224547-us-east-1-5epobvmonaep0n",
    object_key="applicant_data.json",
    output_path=Path("applicant_data_SM.json")
)

Downloaded s3://amazon-sagemaker-313730224547-us-east-1-5epobvmonaep0n/applicant_data.json -> applicant_data_SM.json


In [4]:
from pathlib import Path

p = Path("applicant_data_SM.json")
print("exists:", p.exists())
print("size bytes:", p.stat().st_size if p.exists() else None)

exists: True
size bytes: 2419003


In [5]:
import json

with open("applicant_data_SM.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("type:", type(data))
print("num records:", len(data) if isinstance(data, list) else "NOT A LIST")
print("first record keys:", list(data[0].keys()) if isinstance(data, list) and data else None)

type: <class 'list'>
num records: 4500
first record keys: ['url', 'institution', 'program', 'degree', 'country_of_origin', 'decision', 'notification', 'undergrad_gpa', 'gre_general', 'gre_verbal', 'gre_aw', 'comments', 'date_added', 'start_term', 'notes']


In [6]:
for i in range(3):
    print("\n--- record", i, "---")
    print(data[i])


--- record 0 ---
{'url': 'https://www.thegradcafe.com/result/919377', 'institution': 'Florida State University (FSU)', 'program': 'Communication Sciences And Disorders', 'degree': 'Masters', 'country_of_origin': 'American', 'decision': 'Accepted', 'notification': 'on 15/02/2024 via E-mail', 'undergrad_gpa': '3.40', 'gre_general': '0', 'gre_verbal': '0', 'gre_aw': '0.00', 'comments': None, 'date_added': None, 'start_term': None, 'notes': 'accepted today for online (DL Program)!'}

--- record 1 ---
{'url': 'https://www.thegradcafe.com/result/919378', 'institution': 'Boston University', 'program': 'Sociology', 'degree': 'PhD', 'country_of_origin': 'American', 'decision': 'Rejected', 'notification': 'on 15/02/2024 via E-mail', 'undergrad_gpa': '3.70', 'gre_general': '0', 'gre_verbal': '0', 'gre_aw': '0.00', 'comments': None, 'date_added': None, 'start_term': None, 'notes': 'No GRE. Was a genuinely good fit :( losing hope for this cycle but trying so hard to stay positive. 0a/4r/9p'}

--- 

In [7]:
import pandas as pd

df = pd.DataFrame(data)

print(df.shape)
df.head()

(4500, 15)


,url,institution,program,degree,country_of_origin,decision,notification,undergrad_gpa,gre_general,gre_verbal,gre_aw,comments,date_added,start_term,notes
0,https://www.thegradcafe.com/result/919377,Florida State University (FSU),Communication Sciences And Disorders,Masters,American,Accepted,on 15/02/2024 via E-mail,3.40,0,0,0.00,None,None,None,accepted today for online (DL Program)!
1,https://www.thegradcafe.com/result/919378,Boston University,Sociology,PhD,American,Rejected,on 15/02/2024 via E-mail,3.70,0,0,0.00,None,None,None,No GRE. Was a genuinely good fit :( losing hop...
2,https://www.thegradcafe.com/result/919379,University Of Pittsburgh,Clinical Psychology,PhD,American,Wait listed,on 15/02/2024 via E-mail,0.00,0,0,0.00,None,None,None,IGNORE STATUS: Has anyone received information...
3,https://www.thegradcafe.com/result/919380,Massachusetts Institute of Technology (MIT),Physics,PhD,International,Rejected,on 15/02/2024 via E-mail,0.00,0,0,0.00,None,None,None,
4,https://www.thegradcafe.com/result/919381,Massachusetts Institute of Technology (MIT),Mechanical Engineering,PhD,International,Rejected,on 15/02/2024 via E-mail,3.80,0,0,0.00,None,None,None,"Masters Degree in ME, 2 years of research/work..."


In [8]:
df.columns

Index(['url', 'institution', 'program', 'degree', 'country_of_origin',
       'decision', 'notification', 'undergrad_gpa', 'gre_general',
       'gre_verbal', 'gre_aw', 'comments', 'date_added', 'start_term',
       'notes'],
      dtype='object')

In [9]:
df["decision"].value_counts()

decision
Accepted       1841
Rejected       1791
Wait listed     592
Interview       275
Name: count, dtype: int64